# Bhajan Aabha — Autonomous Cloud Worker

This is the remote GPU worker for the zero-cost autonomous channel. It is intentionally designed so the user's computer is never used.

Run contract: discover devotional trends → select 1–4 opportunities → create original music/visual plan → render → QA → publish to YouTube and Facebook → record results.

**Copyright rule:** never download, remix, pitch-shift, speed-change, or otherwise alter a copyrighted recording to evade detection. Use original generations, public-domain material, or explicitly licensed assets only.


In [ ]:
import os, sys, json, time, subprocess, shutil, hashlib
from pathlib import Path
WORK = Path('/kaggle/working')
OUT = WORK/'output'; OUT.mkdir(exist_ok=True)
print('GPU:', shutil.which('nvidia-smi'))
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'])


In [ ]:
# Runtime dependencies. Heavy ML packages will be pinned in the production version after the smoke test.
subprocess.run([sys.executable,'-m','pip','install','-q','requests','pillow','numpy','moviepy','yt-dlp','google-api-python-client','google-auth-oauthlib','google-auth-httplib2'], check=True)


In [ ]:
# Configuration is supplied through Kaggle Secrets / environment variables.
CHANNEL='Bhajan Aabha'
MAX_VIDEOS=int(os.getenv('MAX_VIDEOS_PER_RUN','4'))
MAX_VIDEOS=max(1,min(4,MAX_VIDEOS))
required=['YOUTUBE_CLIENT_ID','YOUTUBE_CLIENT_SECRET','YOUTUBE_REFRESH_TOKEN','FACEBOOK_PAGE_ID','FACEBOOK_PAGE_ACCESS_TOKEN','YOUTUBE_API_KEY']
missing=[x for x in required if not os.getenv(x)]
print('channel:', CHANNEL, 'target:', MAX_VIDEOS)
print('missing credentials:', missing)


In [ ]:
# Safe smoke-test marker. Production generation is enabled only after this remote GPU path is proven.
state={'channel':CHANNEL,'max_videos':MAX_VIDEOS,'missing_credentials':missing,'worker':'kaggle-gpu','local_compute':False}
(WORK/'worker_state.json').write_text(json.dumps(state,ensure_ascii=False,indent=2))
print(json.dumps(state,ensure_ascii=False,indent=2))
if missing:
    print('SETUP_PENDING — no publishing or paid service is attempted.')
